# **Fase 4: Optimizacion de Hiperparametros**

Estudiante: Maria Camila Navarrete Pinzón

Código: 2294353

Fecha: 08 febrero, 2026

# Notebook 8: Validacion Cruzada

**Objetivo**: Comparar Ridge (L2), Lasso (L1) y ElasticNet

**Conceptos clave**:

**Ridge (L2)**: regParam > 0, elasticNetParam = 0
- Penaliza coeficientes grandes, NO los elimina
  
**Lasso (L1)**: regParam > 0, elasticNetParam = 1
- Puede eliminar features (coeficientes = 0)
 - Combinación de L1 y L2

**Actividades**:

1. Entrenar modelos con diferentes regularizaciones
2. Comparar resultados
3. Identificar el mejor modelo

## 1. Configuración de SparkSession

Se crea una sesión de spark configurada para ejecutarse en modo local, asignando memoria al driver.

In [2]:

from pyspark.sql import SparkSession
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName("SECOP_CrossValidation") \
    .master("spark://spark-master:7077") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/13 23:34:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/02/13 23:34:49 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/02/13 23:35:05 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


## 2. Carga de datos

In [3]:
df = spark.read.parquet("/opt/spark-data/processed/secop_ml_ready.parquet")
df = df.withColumnRenamed("valor_del_contrato_num", "label") \
       .withColumnRenamed("features_pca", "features") \
       .filter(col("label").isNotNull())

train, test = df.randomSplit([0.8, 0.2], seed=42)

print(f"Train: {train.count():,}")
print(f"Test: {test.count():,}")


Train: 41,925


Test: 10,323


## 3. Reto 1: Entender la Regularización

**Pregunta conceptual**: Si usas K=5, responde:

1. ¿En cuántos subconjuntos se dividen los datos de train?
2. ¿Cuántos modelos se entrenan en total?
3. ¿Qué porcentaje de datos se usa para validación en cada iteración?
4. ¿Qué métrica se reporta al final?

**Diagrama mental**:
 ```
# Fold 1: [VAL] [Train] [Train] [Train] [Train]
# Fold 2: [Train] [VAL] [Train] [Train] [Train]
# Fold 3: [Train] [Train] [VAL] [Train] [Train]
# Fold 4: [Train] [Train] [Train] [VAL] [Train]
# Fold 5: [Train] [Train] [Train] [Train] [VAL]
 ```

**¿Por qué es mejor que un simple train/test split?**

1. Subconjuntos: Se divide el conjunto de entrenamiento en 5 subconjuntos (folds).
2. Modelos entrenados: Se entrenan 5 modelos (uno por cada fold).
3. Porcentaje validación: En cada iteración se usa el 20% de los datos para validación y el 80% para entrenamiento.

4. Métrica final Se reporta el promedio de la métrica a lo largo de los 5 folds.:
Ventaja sobre train/test simple:


## 4. Reto 2: Crear el Modelo Base y Evaluador

**Objetivo**: Configurar LinearRegression y RegressionEvaluator.

**Instrucciones**:
1. Crea un modelo de LinearRegression (sin hiperparámetros fijos)
2. Crea un evaluador con la métrica apropiada

In [5]:
lr = LinearRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=1000
)

evaluator = RegressionEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="rmse"
)


## 5. Reto 3: 	Construir ParamGrid de hiperparametros

**Objetivo**: Definir la grilla de hiperparámetros a explorar.

**Instrucciones**:
1. Usa `ParamGridBuilder` para crear combinaciones
2. Incluye al menos regParam y elasticNetParam
3. Calcula cuántas combinaciones hay

**Pregunta**: Si agregas 3 valores de regParam y 3 de elasticNetParam con K=5, ¿cuántos modelos se entrenan en total?


In [6]:
param_grid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0.01, 0.1, 1.0]) \
    .addGrid(lr.elasticNetParam, [0.0, 0.5, 1.0]) \
    .build()

K = 5
print(f"Combinaciones en el grid: {len(param_grid)}")
print(f"Total de modelos a entrenar: {len(param_grid) * K}")


Combinaciones en el grid: 9
Total de modelos a entrenar: 45


Cada combinación de hiperparámetros se evalúa en cada fold, lo que multiplica el costo computacional pero mejora la confiabilidad del resultado.

## 6. Reto 4: 	Configurar CrossValidator (elegir K)

**Objetivo**: Ensamblar el CrossValidator con modelo, grid y evaluador.

**Parámetros clave**:
- `estimator`: El modelo a entrenar
- `estimatorParamMaps`: Las combinaciones de hiperparámetros
- `evaluator`: Cómo evaluar cada modelo
- `numFolds`: Valor de K (típicamente 3, 5 o 10)
- `seed`: Para reproducibilidad

**Pregunta**: ¿Qué valor de K elegirías?
- K=3: Más rápido, menos robusto
- K=5: Balance clásico
- K=10: Más robusto, más lento

**Consideración**: K grande en datasets grandes = MUY costoso



In [7]:
crossval = CrossValidator(
    estimator=lr,
    estimatorParamMaps=param_grid,
    evaluator=evaluator,
    numFolds=5,
    seed=42
)

print("Cross-Validation configurada con K=5")
print(f"Total modelos a entrenar: {len(param_grid) * 5}")


Cross-Validation configurada con K=5
Total modelos a entrenar: 45


K=5 es un balance óptimo entre:
- Robustez estadística
- Costo computacional razonable

## 7. Reto 5: 	Ejecutar CV y analizar metricas por configuracion

**Objetivo**: Entrenar, obtener métricas y encontrar el mejor modelo.

**Instrucciones**:
1. Ejecuta `crossval.fit(train)`
2. Obtén las métricas promedio con `cv_model.avgMetrics`
3. Identifica la mejor configuración
4. Evalúa el mejor modelo en test


In [8]:
print("Entrenando con Cross-Validation...")
cv_model = crossval.fit(train)
print("Cross-validation completada")

avg_metrics = cv_model.avgMetrics
best_metric_idx = avg_metrics.index(min(avg_metrics))

print("\nMétricas promedio por configuración")
for i, metric in enumerate(avg_metrics):
    params = param_grid[i]
    reg = params.get(lr.regParam)
    elastic = params.get(lr.elasticNetParam)
    marker = " <-- MEJOR" if i == best_metric_idx else ""
    print(f"Config {i+1}: λ={reg:.2f}, α={elastic:.1f} -> RMSE={metric:,.2f}{marker}")

best_model = cv_model.bestModel

print("\nMejor modelo")
print(f"regParam: {best_model.getRegParam()}")
print(f"elasticNetParam: {best_model.getElasticNetParam()}")

predictions = best_model.transform(test)
rmse_test = evaluator.evaluate(predictions)
print(f"RMSE en Test: ${rmse_test:,.2f}")


Entrenando con Cross-Validation...


26/02/14 00:27:10 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/02/14 00:27:10 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.VectorBLAS
26/02/14 00:27:11 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK


Cross-validation completada

Métricas promedio por configuración
Config 1: λ=0.01, α=0.0 -> RMSE=17,319,674,295.87
Config 2: λ=0.01, α=0.5 -> RMSE=17,319,674,295.87
Config 3: λ=0.01, α=1.0 -> RMSE=17,319,674,295.86
Config 4: λ=0.10, α=0.0 -> RMSE=17,319,674,295.86
Config 5: λ=0.10, α=0.5 -> RMSE=17,319,674,295.83
Config 6: λ=0.10, α=1.0 -> RMSE=17,319,674,295.80
Config 7: λ=1.00, α=0.0 -> RMSE=17,319,674,295.82
Config 8: λ=1.00, α=0.5 -> RMSE=17,319,674,295.51
Config 9: λ=1.00, α=1.0 -> RMSE=17,319,674,295.20 <-- MEJOR

Mejor modelo
regParam: 1.0
elasticNetParam: 1.0


RMSE en Test: $5,200,424,546.28


Los RMSE obtenidos en las diferentes configuraciones son muy similares porque el valor del target está en una escala extremadamente alta, lo que hace que variaciones pequeñas entre modelos queden prácticamente invisibles en la métrica. Aunque las diferencias son mínimas, se observa una ligera mejora al aumentar la regularización y usar Lasso, lo cual es consistente con la teoría y sugiere que el tuning sí tiene efecto, pero limitado por la naturaleza de los datos y las features utilizadas.

## 8. Reto 6: 	Comparar CV vs simple train/test split

**Objetivo**: Demostrar la ventaja de cross-validation.

**Instrucciones**:
1. Entrena un modelo con los mismos hiperparámetros pero SIN CV
2. Compara el RMSE de test con el modelo de CV
3. ¿Cuál es más confiable?

**Pregunta**: ¿La métrica de CV es más cercana al rendimiento real? ¿Por qué?

In [9]:
lr_simple = LinearRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=100,
    regParam=best_model.getRegParam(),
    elasticNetParam=best_model.getElasticNetParam()
)

model_simple = lr_simple.fit(train)
rmse_simple = evaluator.evaluate(model_simple.transform(test))

print(f"RMSE con CV: ${rmse_test:,.2f}")
print(f"RMSE sin CV: ${rmse_simple:,.2f}")
print(f"Diferencia: ${abs(rmse_test - rmse_simple):,.2f}")


RMSE con CV: $5,200,424,546.28
RMSE sin CV: $5,200,424,546.28
Diferencia: $0.00


La validación cruzada es más confiable, ya que su métrica refleja el comportamiento promedio del modelo y no depende de una sola división aleatoria.

## 9. Bonus 1: 	Experimentar con K=3, K=5, K=10

**Objetivo**: Observar el efecto del número de folds.

**Instrucciones**:
1. Ejecuta CV con K=3, K=5, K=10
2. Compara la métrica promedio del mejor modelo en cada caso
3. Mide el tiempo de ejecución de cada uno

**Pregunta**: ¿Más folds siempre es mejor?

In [10]:
import time

for k in [3, 5, 10]:
    cv_temp = CrossValidator(
        estimator=lr,
        estimatorParamMaps=param_grid,
        evaluator=evaluator,
        numFolds=k,
        seed=42
    )
    start = time.time()
    cv_temp_model = cv_temp.fit(train)
    elapsed = time.time() - start
    best_rmse = min(cv_temp_model.avgMetrics)
    print(f"K={k} | Mejor RMSE: ${best_rmse:,.2f} | Tiempo: {elapsed:.1f}s")


K=3 | Mejor RMSE: $18,150,169,157.54 | Tiempo: 65.8s


K=5 | Mejor RMSE: $17,319,674,295.20 | Tiempo: 55.7s


K=10 | Mejor RMSE: $14,207,181,527.21 | Tiempo: 80.3s


Más folds no siempre es mejor porque mejora estabilidad pero aumenta exponencialmente el tiempo de entrenamiento

## 10. Preguntas de Reflexión

**¿Cuándo usarías K=3 vs K=10?**

*Respuesta:* K=3 en datasets grandes o pruebas rápidas; K=10 cuando el dataset es pequeño y se necesita máxima robustez.

**¿Cross-validation reemplaza la necesidad de un test set?**

*Respuesta:* No. El test set sigue siendo necesario para una evaluación final honesta.

**¿Qué pasa si tu dataset tiene solo 100 registros? ¿Qué K usarías?**

*Respuesta:* K=10 para aprovechar al máximo los datos disponibles.

**¿Es posible hacer CV con time series? ¿Qué cambiaría?**

*Respuesta:* Sí, pero usando validación temporal (rolling window), nunca folds aleatorios.

In [11]:
model_path = "/opt/spark-data/processed/cv_best_model"
best_model.save(model_path)
print(f"Modelo guardado en: {model_path}")


Modelo guardado en: /opt/spark-data/processed/cv_best_model


In [12]:

print("Resumen validación cruzada")
print("Verifica que hayas completado:")
print("  [✓] Entendido el concepto de K-Fold")
print("  [✓] Configurado ParamGrid con hiperparámetros")
print("  [✓] Ejecutado CrossValidator")
print("  [✓] Identificado el mejor modelo")
print("  [✓] Comparado con entrenamiento simple")


Resumen validación cruzada
Verifica que hayas completado:
  [✓] Entendido el concepto de K-Fold
  [✓] Configurado ParamGrid con hiperparámetros
  [✓] Ejecutado CrossValidator
  [✓] Identificado el mejor modelo
  [✓] Comparado con entrenamiento simple


In [13]:
spark.stop()